# Notebook 02 — Biomechanics Analysis

Deep dive into biomechanical features: joint angles, angular velocities,
wrist speed profiles, and make vs miss differences.

**Goal**: Confirm that engineered features capture real biomechanical differences
between successful and unsuccessful shots.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_theme(style='whitegrid')
MAKE_COLOR, MISS_COLOR = '#2ecc71', '#e74c3c'
print('✅ Libraries loaded')

## 1. Load Feature Data

In [ ]:
feat_path = Path('../data/processed/features/biomech_features.csv')
rel_path  = Path('../data/processed/features/release_features.csv')

features_df = pd.read_csv(feat_path)
release_df  = pd.read_csv(rel_path)

print(f'Frame-level features: {len(features_df):,} rows × {len(features_df.columns)} cols')
print(f'Release features    : {len(release_df)} shots × {len(release_df.columns)} cols')
print(f'Make rate           : {release_df["outcome"].mean():.1%}')

## 2. Joint Angle Profiles Over Time

Average angle trajectories aligned by shot phase. Key insight: makes and misses
diverge most strongly near the release frame.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
angle_cols = ['elbow_angle', 'right_knee_angle', 'shoulder_angle', 'trunk_lean']

for ax, col in zip(axes.flatten(), angle_cols):
    if col not in features_df.columns:
        ax.text(0.5, 0.5, f'{col}\nnot available', ha='center', va='center',
               transform=ax.transAxes)
        continue
    
    for outcome, label, color in [(1, 'Make', MAKE_COLOR), (0, 'Miss', MISS_COLOR)]:
        trajectories = []
        for _, grp in features_df[features_df['outcome']==outcome].groupby('video_stem'):
            vals = grp.sort_values('frame')[col].dropna().values
            if len(vals) > 5:
                resampled = np.interp(
                    np.linspace(0, 1, 50),
                    np.linspace(0, 1, len(vals)),
                    vals
                )
                trajectories.append(resampled)
        
        if trajectories:
            arr = np.array(trajectories)
            x = np.linspace(0, 1, 50)
            mean, std = arr.mean(0), arr.std(0)
            ax.plot(x, mean, color=color, linewidth=2.5,
                   label=f'{label} (n={len(trajectories)})')
            ax.fill_between(x, mean-std, mean+std, color=color, alpha=0.12)
    
    ax.axvline(0.6, color='gray', linestyle=':', alpha=0.7, label='~Release')
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Shot phase (normalized)')
    ax.set_ylabel('Angle (°)')
    ax.legend(fontsize=9)

plt.suptitle('Joint Angle Trajectories: Make vs Miss', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Wrist Speed Profile — The Release Signature

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for outcome, label, color in [(1, 'Make', MAKE_COLOR), (0, 'Miss', MISS_COLOR)]:
    trajectories = []
    for _, grp in features_df[features_df['outcome']==outcome].groupby('video_stem'):
        ws = grp.sort_values('frame')['wrist_speed'].fillna(0).values
        if len(ws) > 5:
            trajectories.append(np.interp(
                np.linspace(0, 1, 60),
                np.linspace(0, 1, len(ws)), ws
            ))
    
    if trajectories:
        arr = np.array(trajectories)
        x = np.linspace(0, 1, 60)
        mean, std = arr.mean(0), arr.std(0)
        ax.plot(x, mean, color=color, linewidth=2.5, label=f'{label} (n={len(trajectories)})')
        ax.fill_between(x, mean-std, mean+std, color=color, alpha=0.12)

ax.axvspan(0.5, 0.7, alpha=0.08, color='gold', label='Release window')
ax.set_xlabel('Shot Phase (normalized)', fontsize=12)
ax.set_ylabel('Wrist Speed (norm. units/sec)', fontsize=12)
ax.set_title('Wrist Speed Profile: Make vs Miss\n(Peak = Release Point)', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('Insight: Makes typically have a sharper, higher wrist speed peak = stronger wrist snap')

## 4. Statistical Significance Testing

Are the observed differences between make and miss statistically significant?

In [ ]:
test_features = [
    'release_elbow_angle', 'release_knee_angle', 'release_wrist_height',
    'release_wrist_speed', 'pre_release_knee_bend', 'follow_through_elbow',
    'release_shoulder_tilt', 'release_trunk_lean'
]

results = []
for feat in test_features:
    if feat not in release_df.columns:
        continue
    makes  = release_df[release_df['outcome']==1][feat].dropna()
    misses = release_df[release_df['outcome']==0][feat].dropna()
    
    if len(makes) < 3 or len(misses) < 3:
        continue
    
    t_stat, p_val = stats.ttest_ind(makes, misses)
    cohen_d = (makes.mean() - misses.mean()) / (
        np.sqrt((makes.std()**2 + misses.std()**2) / 2) + 1e-9
    )
    results.append({
        'Feature': feat.replace('_', ' ').title(),
        'Make Mean': f'{makes.mean():.2f}',
        'Miss Mean': f'{misses.mean():.2f}',
        'p-value': f'{p_val:.4f}',
        'Significant': '✅' if p_val < 0.05 else '❌',
        "Cohen's d": f'{abs(cohen_d):.3f}',
        'Effect': 'Large' if abs(cohen_d) > 0.8 else 'Medium' if abs(cohen_d) > 0.5 else 'Small'
    })

results_df = pd.DataFrame(results)
print('STATISTICAL TESTS (t-test, Make vs Miss)')
print('='*80)
print(results_df.to_string(index=False))

## 5. Feature Correlation Heatmap

In [ ]:
from src.modeling.form_scorer import FORM_FEATURES
available = [f for f in FORM_FEATURES if f in release_df.columns] + ['outcome']
corr = release_df[available].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
           center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix (with Outcome)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Interactive Scatter — Explore Feature Pairs

In [ ]:
release_df['Outcome'] = release_df['outcome'].map({1: 'Make', 0: 'Miss'})

if 'release_elbow_angle' in release_df.columns and 'release_wrist_height' in release_df.columns:
    fig = px.scatter(
        release_df,
        x='release_elbow_angle',
        y='release_wrist_height',
        color='Outcome',
        color_discrete_map={'Make': MAKE_COLOR, 'Miss': MISS_COLOR},
        hover_data=['video_stem', 'shot_type', 'pre_release_knee_bend'],
        title='Elbow Angle vs Wrist Height at Release',
        labels={
            'release_elbow_angle': 'Elbow Angle at Release (°)',
            'release_wrist_height': 'Wrist Height (norm.)',
        },
        opacity=0.75,
    )
    # Add ideal zones
    fig.add_vrect(x0=80, x1=105, fillcolor='green', opacity=0.05,
                 annotation_text='Ideal elbow zone', annotation_position='top left')
    fig.update_layout(height=500)
    fig.show()